# Deepfake Detection — End-to-End Colab Notebook (v2)

Run the cells **in order, top to bottom**, every time you start a fresh session. Before running:
1. `Runtime -> Change runtime type -> T4 GPU`
2. Make sure `deepfake_detection.zip` and your dataset zip (e.g. `archive.zip`) are already uploaded somewhere in your Google Drive.

**What's new in this version:**
- Checkpoints are now saved with a unique filename PER EPOCH (`epoch_001_complete.pth`, etc.) so an accidental/fresh run can never silently overwrite further-along progress — this fixed a real incident from an earlier version.
- Training refuses to run on CPU by default (prevents wasted/corrupting runs if the GPU runtime fails to attach).
- You can train on a FRACTION of your dataset (`TRAIN_SUBSET_FRACTION`) to make each epoch fit comfortably within a free-tier Colab GPU session instead of requiring ~3 hours per epoch on the full dataset.
- Uses a NEW checkpoint folder name (`deepfake_ckpts_v2`) so old, potentially corrupted checkpoints from a previous attempt are never accidentally reused.

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Extract the project code
Edit the path below if you uploaded `deepfake_detection.zip` somewhere other than the root of My Drive.

In [ ]:
PROJECT_ZIP_PATH = "/content/drive/MyDrive/deepfake_detection.zip"  # <-- EDIT IF NEEDED

!rm -rf /content/deepfake_detection
!unzip -q "$PROJECT_ZIP_PATH" -d /content/
%cd /content/deepfake_detection
!ls

## Step 3: Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## Step 4: Check GPU is actually attached
Do this BEFORE extracting the dataset or training — no point doing the rest if there's no GPU this session.

In [ ]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU attached this session — go to Runtime > Change runtime type > GPU, then Runtime > Restart session, then re-run from Step 1."

## Step 5: Extract your dataset to LOCAL disk (fast)
Reading directly off Google Drive during training is much slower than local disk. This copies your dataset zip to local Colab storage once per session (needs to be redone every fresh session, since `/content/` is wiped on disconnect).

In [ ]:
DATASET_ZIP_PATH = "/content/drive/MyDrive/archive.zip"  # <-- EDIT THIS

!rm -rf /content/local_dataset
!mkdir -p /content/local_dataset
!unzip -q "$DATASET_ZIP_PATH" -d "/content/local_dataset"
print("Extraction finished. Folder layout (up to 4 levels deep):")
!find /content/local_dataset -maxdepth 4 -type d

## Step 6: Configure dataset path + checkpoint location + (optional) subset fraction

- `DEEPFAKE_DATASET_ROOT`: set this to the folder printed above that directly contains your `Train`/`Validation`/`Test` (or `train`/`val`/`test`) folders.
- `DEEPFAKE_CKPT_DIR` / `DEEPFAKE_REPORTS_DIR`: point at Drive so they survive a disconnect. Uses a NEW `_v2` folder name to avoid ever mixing with old checkpoints from a previous attempt.
- `DEEPFAKE_TRAIN_SUBSET_FRACTION` / `DEEPFAKE_VAL_SUBSET_FRACTION`: set below 1.0 (e.g. `0.3`) to train on a fraction of your data — cuts time-per-epoch roughly proportionally. Leave at `1.0` to use the full dataset.
- `DEEPFAKE_NUM_EPOCHS` / `DEEPFAKE_EARLY_STOPPING_PATIENCE`: tune how long training runs before stopping.

**IMPORTANT: run this cell BEFORE the first time you import `config` anywhere in this session** (the cell after this one is fine — just don't import config earlier than this).

In [ ]:
import os

os.environ["DEEPFAKE_DATASET_ROOT"] = "/content/local_dataset/Dataset"  # <-- EDIT THIS to match Step 5's output

# Uncomment and edit any of these ONLY if your folder names differ from the defaults:
# os.environ["DEEPFAKE_TRAIN_DIRNAME"] = "Train"
# os.environ["DEEPFAKE_VAL_DIRNAME"] = "Validation"
# os.environ["DEEPFAKE_TEST_DIRNAME"] = "Test"
# os.environ["DEEPFAKE_REAL_DIRNAME"] = "Real"
# os.environ["DEEPFAKE_FAKE_DIRNAME"] = "Fake"

# NEW folder names — keeps this run's checkpoints completely separate from any
# earlier attempt's (potentially corrupted) checkpoints:
os.environ["DEEPFAKE_CKPT_DIR"] = "/content/drive/MyDrive/deepfake_ckpts_v2"
os.environ["DEEPFAKE_REPORTS_DIR"] = "/content/drive/MyDrive/deepfake_reports_v2"

# OPTIONAL: train on a subset of the data to fit within a realistic Colab
# free-tier GPU session. Comment these out (or set to "1.0") to use 100% of
# your data instead.
os.environ["DEEPFAKE_TRAIN_SUBSET_FRACTION"] = "0.3"
os.environ["DEEPFAKE_VAL_SUBSET_FRACTION"] = "0.3"

# OPTIONAL: tune total training length.
os.environ["DEEPFAKE_NUM_EPOCHS"] = "10"
os.environ["DEEPFAKE_EARLY_STOPPING_PATIENCE"] = "3"

print("Environment variables set.")

## Step 7: Sanity-check the dataset loads correctly
This is the FIRST time `config` gets imported this session — make sure Step 6 ran first.

In [ ]:
from config import cfg
from utils.dataset import DeepfakeFaceDataset

print("TRAIN_REAL_DIR:", cfg.TRAIN_REAL_DIR)
print("TRAIN_FAKE_DIR:", cfg.TRAIN_FAKE_DIR)
print("VAL_REAL_DIR:  ", cfg.VAL_REAL_DIR)
print("VAL_FAKE_DIR:  ", cfg.VAL_FAKE_DIR)
print("NUM_EPOCHS:    ", cfg.NUM_EPOCHS)
print("TRAIN_SUBSET_FRACTION:", cfg.TRAIN_SUBSET_FRACTION)

ds = DeepfakeFaceDataset(
    cfg.TRAIN_REAL_DIR, cfg.TRAIN_FAKE_DIR,
    use_face_detection=False, subset_fraction=cfg.TRAIN_SUBSET_FRACTION,
)
print("\nTotal training images that will be used this run:", len(ds))

## Step 8: Train the model

- First run this session: prints `"No existing checkpoint found — starting fresh from epoch 1."`
- Every run after a disconnect: prints `"Found checkpoint at .../epoch_NNN_(mid|complete).pth, resuming..."` and continues from there automatically.
- If this ever refuses to run with a CPU error message, go back to Step 4 — the GPU didn't attach this session.

In [ ]:
!python train.py

## Step 9: Check your checkpoints on Drive
Run this anytime to see what's been saved so far — useful to confirm progress is safe before ending a session.

In [ ]:
!ls -la /content/drive/MyDrive/deepfake_ckpts_v2/

## Step 10: Evaluate on the test set

In [ ]:
!python test.py

## Step 11: Predict on a single image
Upload a test image to Drive first, then edit `IMAGE_PATH` below.

In [ ]:
IMAGE_PATH = "/content/drive/MyDrive/test_images/sample.jpg"  # <-- EDIT THIS

from predict import predict_image
result = predict_image(IMAGE_PATH)
print(result)

In [ ]:
from IPython.display import IFrame
IFrame(result["report_path"], width=700, height=900)

## Step 12: Predict on a video
Upload a test video to Drive first, then edit `VIDEO_PATH` below.

In [ ]:
VIDEO_PATH = "/content/drive/MyDrive/test_videos/sample.mp4"  # <-- EDIT THIS

from video_predict import predict_video
video_result = predict_video(VIDEO_PATH)
print(video_result)

In [ ]:
from IPython.display import IFrame
IFrame(video_result["report_path"], width=700, height=900)

## Troubleshooting
- **`RuntimeError: No images found`** -> `DATASET_ROOT` in Step 6 is wrong. Re-check Step 5's folder listing.
- **Training refuses to run, says CPU detected** -> go back to Step 4, fix the GPU runtime, restart session, re-run from Step 1.
- **CUDA out of memory** -> lower `BATCH_SIZE` in `config.py` (try 8 or 4), restart session, re-run from Step 3.
- **`ModuleNotFoundError: No module named 'config'`** -> your working directory reset after a restart. Re-run Step 2's `%cd` command.
- **Session disconnected mid-training** -> just reconnect and re-run Steps 1 -> 8 in order (Steps 1-7 are quick, ~2-5 minutes). Training will automatically resume from the most advanced `epoch_NNN_*.pth` checkpoint found — it CANNOT be overwritten by an older/worse run, since each epoch has its own unique filename now.
- **Want to start completely over** -> change `DEEPFAKE_CKPT_DIR` in Step 6 to a brand new folder name (e.g. `deepfake_ckpts_v3`) so old checkpoints are never touched.